In [1]:
%env CUDA_VISIBLE_DEVICES=2

env: CUDA_VISIBLE_DEVICES=2


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = "cpu"
model_path = "/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2"  # "/models/huggingface/meta-llama/llama-2-7b-chat-hf"
model = AutoModelForCausalLM.from_pretrained(
    model_path, device_map=device, torch_dtype=torch.bfloat16
).eval()
tokenizer = AutoTokenizer.from_pretrained(model_path)
embed = model.get_input_embeddings()
if not tokenizer.pad_token:
    if tokenizer.eos_token:
        tokenizer.add_special_tokens(
            {
                "pad_token": tokenizer.eos_token,
            }
        )
    else:
        tokenizer.add_special_tokens({"pad_token": "<PAD>"})
        model.resize_token_embeddings(model.config.vocab_size + 1)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
from vllm import LLM, SamplingParams

llm = LLM(model="/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2")

INFO 07-27 16:20:29 llm_engine.py:176] Initializing an LLM engine (v0.5.2) with config: model='/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2', speculative_config=None, tokenizer='/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None), seed=0, served_model_name=/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2, use_v2_block_manager=False, enable_prefix_caching=False)
INFO 07-27 16:20:30 model_runner.py:723] Starti

Loading safetensors checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

INFO 07-27 16:20:32 model_runner.py:737] Loading model weights took 13.4966 GB
INFO 07-27 16:20:37 gpu_executor.py:102] # GPU blocks: 12854, # CPU blocks: 2048
INFO 07-27 16:20:41 model_runner.py:1025] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 07-27 16:20:41 model_runner.py:1029] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 07-27 16:20:57 model_runner.py:1226] Graph capturing finished in 16 secs.


In [4]:
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "My favorite book of all time is",
    "When I wake up in the morning, the first thing I do is",
    "The best vacation I ever took was to",
    "If I could have any superpower, it would be",
    "One thing on my bucket list is",
    "My go-to comfort food is",
    "The most inspiring person in history, in my opinion, is",
    "When I was a child, I wanted to grow up to be",
    "If I could live in any period of history, I would choose",
    "A skill I've always wanted to learn but haven't yet is",
    "The most memorable movie quote for me is",
    "In my free time, I like to",
    "If I could meet any fictional character, I would choose",
    "The last dream I remember having was about",
    "My favorite season of the year is",
    "One thing I wish more people knew about me is",
    "The most delicious meal I've ever had was",
    "If I could travel anywhere in the world right now, I would go to",
    "A hobby I have that most people don't know about is",
    "The best piece of advice I've ever received is",
]
sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=100)

In [5]:
inputs_embeds = []
for prompt in prompts:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        input_embeds = embed(input_ids).squeeze(0)
    print(input_embeds.shape)
    inputs_embeds.append(input_embeds)

torch.Size([6, 4096])
torch.Size([8, 4096])
torch.Size([8, 4096])
torch.Size([15, 4096])
torch.Size([9, 4096])
torch.Size([12, 4096])
torch.Size([8, 4096])
torch.Size([8, 4096])
torch.Size([14, 4096])
torch.Size([14, 4096])
torch.Size([14, 4096])
torch.Size([16, 4096])
torch.Size([9, 4096])
torch.Size([9, 4096])
torch.Size([13, 4096])
torch.Size([9, 4096])
torch.Size([8, 4096])
torch.Size([11, 4096])
torch.Size([11, 4096])
torch.Size([16, 4096])
torch.Size([15, 4096])
torch.Size([12, 4096])


In [6]:
llm.generate(prompts, sampling_params)

Processed prompts: 100%|██████████| 22/22 [00:02<00:00,  8.64it/s, est. speed input: 96.25 toks/s, output: 864.31 toks/s]


[RequestOutput(request_id=0, prompt='Hello, my name is', prompt_token_ids=[1, 22557, 28725, 586, 1141, 349], prompt_embeds_shape=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text=" Marlene and I'm 56 years old.\n\nI have been a member of the Quirky Quilters Guild since 2014. I have enjoyed meeting so many wonderful people through the Guild and I have learned so much from them.\n\nI live in Clyde, Texas, with my husband, 2 teenage daughters, and 3 dogs. We have a small farm where we raise cattle, goats, chickens, and", token_ids=(1471, 28621, 304, 315, 28742, 28719, 28705, 28782, 28784, 1267, 1571, 28723, 13, 13, 28737, 506, 750, 264, 4292, 302, 272, 2332, 361, 4845, 2332, 309, 1532, 420, 1678, 1854, 28705, 28750, 28734, 28740, 28781, 28723, 315, 506, 10306, 5283, 579, 1287, 8590, 905, 1059, 272, 420, 1678, 304, 315, 506, 5996, 579, 1188, 477, 706, 28723, 13, 13, 28737, 2943, 297, 334, 346, 450, 28725, 7826, 28725, 395, 586, 5581, 28725, 28705, 28750, 10799, 465, 19244

In [7]:
llm.generate({"prompt_embeds": inputs_embeds[0]}, sampling_params)

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.20s/it, est. speed input: 0.00 toks/s, output: 45.52 toks/s]


[RequestOutput(request_id=22, prompt=None, prompt_token_ids=[], prompt_embeds_shape=(6, 4096), prompt_logprobs=None, outputs=[CompletionOutput(index=0, text='Tina. I am a full-time writer and I have written 2 books, a cookbook and a memoir, as well as hundreds of articles for various publications. I am also a certified yoga instructor and I teach classes, lead workshops and retreats. I am a wife, a mother of two, a homeowner, a dog lover, a vegetarian and a coffee addict. I live in the San Francisco Bay Area, but I grew up in the Pacific Northwest.', token_ids=(320, 1380, 28723, 315, 837, 264, 2173, 28733, 1536, 6953, 304, 315, 506, 4241, 28705, 28750, 4796, 28725, 264, 4600, 3521, 304, 264, 1626, 7236, 28725, 390, 1162, 390, 10524, 302, 10437, 354, 4118, 23311, 28723, 315, 837, 835, 264, 20654, 21615, 27679, 304, 315, 3453, 6709, 28725, 1736, 26329, 304, 18030, 28713, 28723, 315, 837, 264, 4285, 28725, 264, 3057, 302, 989, 28725, 264, 1611, 10349, 28725, 264, 3914, 19568, 28725, 264, 

In [8]:
llm.generate(
    [{"prompt_embeds": input_embeds} for input_embeds in inputs_embeds], sampling_params
)

Processed prompts: 100%|██████████| 22/22 [00:02<00:00,  8.56it/s, est. speed input: 0.00 toks/s, output: 855.90 toks/s]


[RequestOutput(request_id=23, prompt=None, prompt_token_ids=[], prompt_embeds_shape=(6, 4096), prompt_logprobs=None, outputs=[CompletionOutput(index=0, text='Tara. I\'m an assistant professor of English at the University of Missouri St. Louis. I teach courses on American literature and culture, and my research focuses on Native American literatures and representations of Native Americans in American literature and culture. I\'m currently working on a book project on Native American writing and urbanism. Today, I want to talk to you about one of my favorite Native American writers, Leslie Marmon Silko, and her most famous novel, "Ceremony', token_ids=(320, 2923, 28723, 315, 28742, 28719, 396, 13892, 12192, 302, 4300, 438, 272, 2900, 302, 20611, 662, 28723, 6698, 28723, 315, 3453, 12318, 356, 2556, 11354, 304, 5679, 28725, 304, 586, 3332, 21165, 356, 16024, 2556, 3937, 2863, 304, 23384, 302, 16024, 8165, 297, 2556, 11354, 304, 5679, 28723, 315, 28742, 28719, 5489, 2739, 356, 264, 1820, 2

In [9]:
from random import shuffle

mixed_inputs = [{"prompt_embeds": input_embeds} for input_embeds in inputs_embeds[:10]] + prompts[10:]
shuffle(mixed_inputs)
llm.generate(
    mixed_inputs,
    sampling_params,
)

Processed prompts: 100%|██████████| 22/22 [00:02<00:00,  8.62it/s, est. speed input: 56.00 toks/s, output: 861.53 toks/s]


[RequestOutput(request_id=45, prompt='My favorite season of the year is', prompt_token_ids=[1, 1984, 6656, 3302, 302, 272, 879, 349], prompt_embeds_shape=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text=' Fall, so you know I was excited to visit the Chateau de Chambord during our trip to France.  This magnificent castle was built between 1519 and 1547 for King Francis I as a hunting lodge.  Chambord is the largest chateau in the Loire Valley and is recognized as a UNESCO World Heritage Site.  The castle is located in the Sologne Forest, which is famous for its wild game, including', token_ids=(12168, 28725, 579, 368, 873, 315, 403, 9534, 298, 3251, 272, 689, 380, 581, 340, 689, 1379, 556, 1938, 813, 6596, 298, 4843, 28723, 28705, 851, 27574, 19007, 403, 4429, 1444, 28705, 28740, 28782, 28740, 28774, 304, 28705, 28740, 28782, 28781, 28787, 354, 4041, 6617, 315, 390, 264, 16268, 305, 13167, 28723, 28705, 689, 1379, 556, 349, 272, 7639, 484, 380, 581, 297, 272, 7300, 53

In [10]:
%%sh

curl http://localhost:8000/v1/models

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   559  100   559    0     0   136k      0 --:--:-- --:--:-- --:--:--  136k


{"object":"list","data":[{"id":"/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2","object":"model","created":1721857051,"owned_by":"vllm","root":"/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2","parent":null,"max_model_len":32768,"permission":[{"id":"modelperm-d74cecde53d54e9d833dee57872dbca3","object":"model_permission","created":1721857051,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [11]:
import requests

url = "http://localhost:8000/v1/completions"
requests.post(
    url,
    json={
        "model": "/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2",
        "prompt": prompts,
        "max_tokens": 7,
        "temperature": 0,
    },
).json()

{'id': 'cmpl-910d75121c0740098e4beef82c03d2b1',
 'object': 'text_completion',
 'created': 1721857059,
 'model': '/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2',
 'choices': [{'index': 0,
   'text': ' Katie and I am a ',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 1,
   'text': ' the commander in chief of the military',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 2,
   'text': ' The Great Gatsby by F',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 3,
   'text': ' check my phone. I know,',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 4,
   'text': ' Hawaii. I went with my family',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 5,
   'text': ' the ability to teleport. I',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 6,
   'text

In [12]:
import io
from base64 import b64encode

prompt_embeds = []
for input_embeds in inputs_embeds:
    buff = io.BytesIO()
    torch.save(input_embeds.detach().cpu(), buff)
    prompt_embeds.append(b64encode(buff.getvalue()).decode("utf-8"))

url = "http://localhost:8000/v1/completions"
requests.post(
    url,
    json={
        "model": "/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2",
        "prompt_embeds": prompt_embeds,
        "max_tokens": 7,
        "temperature": 0,
    },
).json()

{'id': 'cmpl-accb95e119a0457ea240e1466c4648b9',
 'object': 'text_completion',
 'created': 1721857097,
 'model': '/models/huggingface/mistralai/Mistral-7B-Instruct-v0.2',
 'choices': [{'index': 0,
   'text': 'Katie and I am a ',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 1,
   'text': 'the commander in chief of the military',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 2,
   'text': 'The Great Gatsby by F',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 3,
   'text': 'check my phone. I know,',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 4,
   'text': 'Hawaii. I went with my family',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 5,
   'text': 'the ability to teleport. I',
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None},
  {'index': 6,
   'text': 'to